# Image Text Alcohol Classification

Classify images as alcohol or non-alcohol using OCR text. 
Run from the repository root or `src/` with the environment in `README.md`. 
Data: `images_data/` or `test/images_test/`.  
Outputs: `results/` or `test/results_test/`.  
Model caches: `.cache/`. 

In [ ]:
test_mode = False  # Use True for the small test dataset.

# Setup

In [ ]:
import hashlib
import json
import os
import re
import sys
import unicodedata
from collections import Counter, defaultdict
from difflib import get_close_matches
from pathlib import Path

import cmudict
import easyocr
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from matplotlib.ticker import MaxNLocator, StrMethodFormatter
from PIL import Image, ImageOps
from scipy import sparse
from sentence_transformers import SentenceTransformer
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    auc,
    classification_report,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
    silhouette_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import normalize
from tqdm.auto import tqdm
from xgboost import XGBClassifier

# Resolve shared paths when the notebook runs from the root or src/.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "src":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.alcohol_lists import ALCOHOL_BRANDS, ALCOHOL_DRINKS


CACHE_DIR = PROJECT_ROOT / ".cache"
if test_mode:
    DATA_DIR = PROJECT_ROOT / "test" / "images_test"
    RESULTS_DIR = PROJECT_ROOT / "test" / "results_test"
else:
    DATA_DIR = PROJECT_ROOT / "images"
    RESULTS_DIR = PROJECT_ROOT / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_RESULTS_DIR = RESULTS_DIR / "data"
OCR_RESULTS_DIR = RESULTS_DIR / "ocr"
DATA_RESULTS_DIR.mkdir(exist_ok=True)
OCR_RESULTS_DIR.mkdir(exist_ok=True)

pd.options.display.float_format = "{:,.2f}".format
plt.rcParams.update(
    {"font.size": 16, "axes.titlesize": 20, "axes.labelsize": 18}
)

# Limit CPU threads to avoid excessive overhead on these small images.
torch.set_num_threads(4)

## Utils

In [ ]:
def save_figure(fig, output_path):
    """Save a figure before displaying it.

    Args:
        fig (matplotlib.figure.Figure): Figure to save and display.
        output_path (Path): Destination inside the results directory.
    """
    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.show()

# Data

## Load Dataset

In [ ]:
def load_image_dataset(data_dir):
    """Load image paths, folder labels, and basic image metadata.

    Args:
        data_dir (Path): Directory containing alcohol and non_alcohol folders.

    Returns:
        pd.DataFrame: One row per image with its path, label, and metadata.
    """
    records = []
    image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

    for label in ("alcohol", "non_alcohol"):
        for image_path in sorted((data_dir / label).iterdir()):
            if image_path.suffix.lower() not in image_extensions:
                continue
            with Image.open(image_path) as image:
                image.load()
                width, height = ImageOps.exif_transpose(image).size
                records.append(
                    {
                        "file_name": image_path.name,
                        "image_path": str(image_path),
                        "label": label,
                        "width": width,
                        "height": height,
                        "format": image.format,
                        "mode": image.mode,
                        "file_size_kb": image_path.stat().st_size / 1024,
                    }
                )

    return pd.DataFrame(records)

In [ ]:
dataset = load_image_dataset(DATA_DIR)

## Dataset Statistics

In [ ]:
class_summary = dataset.groupby("label").size().rename("image_count").to_frame()
class_summary["percentage"] = class_summary["image_count"] / len(dataset) * 100
image_statistics = dataset[["width", "height", "file_size_kb"]].describe().T

print(f"Total images: {len(dataset):,}")
print(f"Total file size: {dataset['file_size_kb'].sum() / 1024:,.2f} MB")
display(
    class_summary.style.format(
        {"image_count": "{:,.0f}", "percentage": "{:,.2f}"}
    )
)
display(image_statistics)

dataset.to_csv(DATA_RESULTS_DIR / "image_metadata.csv", index=False)
class_summary.to_csv(DATA_RESULTS_DIR / "class_summary.csv")
image_statistics.to_csv(DATA_RESULTS_DIR / "image_statistics.csv")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].bar(
    class_summary.index,
    class_summary["image_count"],
    color=["#4878A8", "#59A14F"],
)
axes[0].set(title="Images per Class", xlabel="Class", ylabel="Image count")
axes[0].yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
for position, count in enumerate(class_summary["image_count"]):
    axes[0].text(
        position, count, f"{count:,}", ha="center", va="bottom", fontsize=18
    )
axes[0].set_ylim(0, class_summary["image_count"].max() * 1.15)

axes[1].hist(dataset["width"], bins=15, alpha=0.65, label="Width")
axes[1].hist(dataset["height"], bins=15, alpha=0.65, label="Height")
axes[1].set(title="Image Dimensions", xlabel="Pixels", ylabel="Image count")
axes[1].xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
axes[1].yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
axes[1].legend()
save_figure(fig, DATA_RESULTS_DIR / 'dataset_statistics.png')

# Show the first two images in each class for a reproducible visual check.
examples = dataset.groupby("label", sort=True).head(2)
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
for axis, row in zip(axes.flat, examples.itertuples(index=False)):
    with Image.open(row.image_path) as image:
        axis.imshow(ImageOps.exif_transpose(image).convert("RGB"))
    axis.set_title(f"{row.label} | {row.file_name}", fontsize=18)
    axis.axis("off")
save_figure(fig, DATA_RESULTS_DIR / 'sample_images.png')

# Extract Text

Use English EasyOCR on the CPU.  
Retry low-quality text with mirroring, contrast enhancement, or rotation.  
Select retries using confidence and vocabulary coverage.  
Reuse OCR only when image contents and settings match the cache.  

## OCR

In [ ]:
MIN_ENGLISH_RATIO = 0.80
MIN_OCR_CONFIDENCE = 0.80
MIN_SCORE_GAIN = 0.05
MIN_TEXT_SCORE = 0.20
OCR_PIPELINE_VERSION = 2
OCR_OPTIONS = {
    "decoder": "greedy",
    "min_size": 10,
    "width_ths": 0.30,
    "contrast_ths": 0.30,
    "adjust_contrast": 0.70,
    "rotation_info": [180],
}
OCR_MAGNIFICATION = 2.0
BEVERAGE_WORDS = {
    "absolut", "bacardi", "budweiser", "heineken", "jagermeister", "smirnoff",
    "matcha", "kombucha", "yerba", "espresso", "cappuccino", "latte",
    "coffee", "tea", "juice", "lemonade", "milk", "water", "chocolate",
}
ALCOHOL_TERMS = set(ALCOHOL_BRANDS) | set(ALCOHOL_DRINKS)
OCR_RETRY_COLUMNS = {
    "english_word_ratio", "flip_attempted", "selected_orientation",
    "original_text", "original_mean_confidence", "original_english_word_ratio",
    "flipped_text", "flipped_mean_confidence", "flipped_english_word_ratio",
    "quality_score", "selected_preprocessing", "enhancement_attempted",
    "enhanced_text", "enhanced_mean_confidence", "enhanced_english_word_ratio",
    "ocr_cache_key",
}

MIN_WORD_SIMILARITY = 0.90
MIN_CORRECTION_MARGIN = 0.03
NUMBER_TO_LETTER = {
    "0": "o", "1": "il", "2": "z", "3": "e", "4": "a",
    "5": "s", "6": "g", "7": "t", "8": "bs", "9": "g",
}
LETTER_CONFUSIONS = {"i": "l", "o": "q"}
COMMON_PHRASES = {"get ready", "don't miss", "limited edition", "check out"}
COMMON_TEXT_WORDS = {
    "dont", "don't", "miss", "limited", "edition", "nightlife", "event",
    "get", "ready", "for", "check", "out", "grab", "now", "try", "enjoy",
    "smooth", "premium", "imported", "menu", "special", "offer", "party",
    "reserve", "flavor", "taste", "celebration", "explore", "experience",
    "discount", "bar", "new",
}

In [ ]:
def load_english_vocabulary(extra_words):
    """Load a shared vocabulary for OCR scoring and text cleaning.

    Args:
        extra_words (set[str]): Additional beverage and brand names.

    Returns:
        set[str]: Normalized dictionary words and domain-term components.
    """
    vocabulary = {word.casefold() for word in cmudict.words()}
    for term in set(extra_words) | ALCOHOL_TERMS:
        vocabulary.update(re.findall(r"[^\W_]+(?:'[^\W_]+)*", term.casefold()))
    return vocabulary


def calculate_english_word_ratio(text, english_words):
    """Measure dictionary coverage without changing the transcription.

    Args:
        text (str): OCR text to evaluate.
        english_words (set[str]): Recognized lowercase words.

    Returns:
        float: Recognized-word fraction, or zero for text without words.
    """
    text = text.casefold().replace("’", "'").replace("‘", "'")
    tokens = re.findall(r"[^\W_]+(?:'[^\W_]+)*", text)
    words = [token for token in tokens if not token.isnumeric()]
    return sum(word in english_words for word in words) / len(words) if words else 0.0


def create_ocr_reader(model_dir):
    """Initialize one English EasyOCR reader on the CPU.

    Args:
        model_dir (Path): Directory containing downloaded model weights.

    Returns:
        easyocr.Reader: Reader shared across all images.
    """
    return easyocr.Reader(
        ["en"], gpu=False, model_storage_directory=str(model_dir),
        user_network_directory=str(model_dir / "user_network"), verbose=True,
    )


def summarize_ocr_attempt(detections, english_words):
    """Order text boxes by line and summarize an OCR attempt.

    Args:
        detections (list): Bounding-box, text, and confidence tuples.
        english_words (set[str]): Recognized lowercase words.

    Returns:
        dict: Transcription, coverage, character-weighted confidence, and boxes.
    """
    detections = [item for item in detections if item[1].strip()]
    lines = []
    by_height = sorted(detections, key=lambda item: np.mean(item[0], axis=0)[1])
    for box, text, confidence in by_height:
        points = np.asarray(box, dtype=float)
        center_x, center_y = points.mean(axis=0)
        height = max(np.ptp(points[:, 1]), 1.0)
        item = (center_x, center_y, height, box, text.strip(), float(confidence))
        same_line = lines and (
            abs(center_y - np.mean([entry[1] for entry in lines[-1]]))
            <= height * 0.6
        )
        if same_line:
            lines[-1].append(item)
        else:
            lines.append([item])
    ordered = [
        item for line in lines for item in sorted(line, key=lambda item: item[0])
    ]
    text = " ".join(item[4] for item in ordered)
    confidences = [item[5] for item in ordered]
    lengths = [max(len(item[4].replace(" ", "")), 1) for item in ordered]
    angles = [
        np.degrees(np.arctan2(box[1][1] - box[0][1], box[1][0] - box[0][0]))
        for box, _, _ in detections
    ]
    angles = [angle for angle in angles if abs(angle) <= 45]
    return {
        "extracted_text": text,
        "word_count": len(text.split()),
        "recognized_word_count": sum(
            token in english_words
            for token in re.findall(r"[^\W_]+(?:'[^\W_]+)*", text.casefold())
            if not token.isnumeric()
        ),
        "text_box_count": len(ordered),
        "mean_confidence": (
            np.average(confidences, weights=lengths) if ordered else np.nan
        ),
        "english_word_ratio": calculate_english_word_ratio(text, english_words),
        "text_angle": float(np.median(angles)) if angles else 0.0,
        "text_boxes": json.dumps([np.asarray(item[3]).tolist() for item in ordered]),
    }


def score_ocr_attempt(attempt):
    """Combine confidence and vocabulary coverage for attempt selection.

    Args:
        attempt (dict): Text, confidence, and English-word ratio for an attempt.

    Returns:
        float: Quality score, or -1 for an empty transcription.
    """
    if not attempt["extracted_text"].strip():
        return -1.0
    confidence = attempt["mean_confidence"]
    confidence = 0.0 if pd.isna(confidence) else confidence
    return 0.45 * confidence + 0.55 * attempt["english_word_ratio"]


def is_better_ocr_attempt(candidate, current, require_language=False):
    """Require a meaningful quality gain without losing recognized words.

    Args:
        candidate (dict): Alternative OCR attempt.
        current (dict): Currently selected OCR attempt.
        require_language (bool): Require vocabulary support for mirror selection.

    Returns:
        bool: Whether the alternative should replace the current result.
    """
    if not current["extracted_text"] and score_ocr_attempt(candidate) < MIN_TEXT_SCORE:
        return False
    current_known = current["recognized_word_count"]
    candidate_known = candidate["recognized_word_count"]
    language_supported = (
        candidate["english_word_ratio"] > 0
        and candidate["english_word_ratio"] >= current["english_word_ratio"]
    )
    return (
        (not require_language or language_supported)
        and score_ocr_attempt(candidate) > score_ocr_attempt(current) + MIN_SCORE_GAIN
        and candidate_known >= current_known * 0.8
    )


def read_ocr_attempt(image, reader, english_words, magnification=1.0):
    """Keep native color OCR as a baseline and enhance grayscale on retries.

    Args:
        image (PIL.Image.Image): EXIF-oriented image or a retry variant.
        reader (easyocr.Reader): Shared OCR reader.
        english_words (set[str]): Vocabulary for scoring.
        magnification (float): Detection magnification factor.

    Returns:
        dict: Summarized OCR attempt.
    """
    if magnification == 1.0:
        detections = reader.readtext(
            np.array(image.convert("RGB")), detail=1, paragraph=False,
        )
    else:
        detections = reader.readtext(
            np.array(image.convert("L")), detail=1, paragraph=False,
            mag_ratio=magnification, **OCR_OPTIONS,
        )
    return summarize_ocr_attempt(detections, english_words)


def extract_image_text(
    dataset, reader, english_words, min_english_ratio=0.80, min_confidence=0.80
):
    """Read images and retry mirrors, contrast enhancement, and deskewing.

    Args:
        dataset (pd.DataFrame): Image metadata containing image_path.
        reader (easyocr.Reader): Shared OCR reader.
        english_words (set[str]): Shared scoring vocabulary.
        min_english_ratio (float): Coverage threshold for skipping retries.
        min_confidence (float): Confidence threshold for skipping retries.

    Returns:
        pd.DataFrame: Metadata, selected text, and original/retry diagnostics.
    """
    records = []
    for row in tqdm(dataset.itertuples(index=False), total=len(dataset), desc="OCR"):
        with Image.open(row.image_path) as image:
            image = ImageOps.exif_transpose(image).convert("RGB")
        original = read_ocr_attempt(image, reader, english_words)
        selected = original
        orientation = "original"
        preprocessing = "native"
        flipped = None
        enhanced = None
        flip_attempted = not (
            original["english_word_ratio"] >= min_english_ratio
            and original["mean_confidence"] >= min_confidence
        )
        if flip_attempted:
            flipped = read_ocr_attempt(ImageOps.mirror(image), reader, english_words)
            if is_better_ocr_attempt(flipped, selected, require_language=True):
                selected, orientation = flipped, "flipped"
        enhancement_attempted = not (
            selected["english_word_ratio"] >= min_english_ratio
            and selected["mean_confidence"] >= min_confidence
        )
        if enhancement_attempted:
            retry_image = ImageOps.mirror(image) if orientation == "flipped" else image
            enhanced = read_ocr_attempt(
                ImageOps.autocontrast(retry_image.convert("L")), reader, english_words,
                OCR_MAGNIFICATION,
            )
            if is_better_ocr_attempt(enhanced, selected):
                selected, preprocessing = enhanced, "contrast_scaled"
            angle = selected["text_angle"]
            angles = [angle] if abs(angle) >= 5 else []
            if not selected["extracted_text"]:
                angles = [-30, 30]
            for angle in angles:
                deskewed = retry_image.rotate(angle, expand=True, fillcolor="white")
                attempt = read_ocr_attempt(
                    deskewed, reader, english_words, OCR_MAGNIFICATION
                )
                if is_better_ocr_attempt(attempt, selected):
                    selected, preprocessing = attempt, f"deskew_{angle:.1f}"
                if is_better_ocr_attempt(attempt, enhanced):
                    enhanced = attempt
        records.append({
            **selected,
            "quality_score": score_ocr_attempt(selected),
            "flip_attempted": flip_attempted,
            "selected_orientation": orientation,
            "selected_preprocessing": preprocessing,
            "enhancement_attempted": enhancement_attempted,
            **ocr_attempt_diagnostics(original, "original"),
            **ocr_attempt_diagnostics(flipped, "flipped"),
            **ocr_attempt_diagnostics(enhanced, "enhanced"),
        })
    return pd.concat([dataset.reset_index(drop=True), pd.DataFrame(records)], axis=1)


def create_ocr_cache_key(dataset, english_words):
    """Fingerprint image contents, vocabulary, and OCR settings.

    Args:
        dataset (pd.DataFrame): Current image paths in extraction order.
        english_words (set[str]): Current scoring vocabulary.

    Returns:
        str: SHA-256 fingerprint stored with the OCR rows.
    """
    settings = {
        "version": OCR_PIPELINE_VERSION,
        "options": OCR_OPTIONS,
        "magnification": OCR_MAGNIFICATION,
        "min_confidence": MIN_OCR_CONFIDENCE,
        "min_english_ratio": MIN_ENGLISH_RATIO,
        "min_score_gain": MIN_SCORE_GAIN,
        "min_text_score": MIN_TEXT_SCORE,
        "vocabulary": sorted(english_words),
    }
    fingerprint = hashlib.sha256(json.dumps(settings, sort_keys=True).encode())
    for row in dataset.itertuples(index=False):
        fingerprint.update(str((row.image_path, row.label)).encode())
        fingerprint.update(hashlib.sha256(Path(row.image_path).read_bytes()).digest())
    return fingerprint.hexdigest()


def load_or_extract_ocr(dataset, results_path, model_dir, english_words):
    """Reuse matching cached results, or extract and save fresh OCR rows.

    Args:
        dataset (pd.DataFrame): Current image metadata.
        results_path (Path): Test/full results CSV path.
        model_dir (Path): Directory containing OCR weights.
        english_words (set[str]): Shared scoring vocabulary.

    Returns:
        pd.DataFrame: Current metadata and OCR results.
    """
    cache_key = create_ocr_cache_key(dataset, english_words)
    if results_path.is_file():
        numeric_columns = [
            "mean_confidence", "original_mean_confidence", "flipped_mean_confidence",
            "flipped_english_word_ratio", "enhanced_mean_confidence",
            "enhanced_english_word_ratio",
        ]
        cached = pd.read_csv(
            results_path, keep_default_na=False,
            dtype={column: str for column in (
                "extracted_text", "original_text", "flipped_text", "enhanced_text",
            )},
            na_values={column: [""] for column in numeric_columns},
        )
        if (
            OCR_RETRY_COLUMNS.issubset(cached.columns)
            and cached["image_path"].tolist() == dataset["image_path"].tolist()
            and cached["ocr_cache_key"].eq(cache_key).all()
        ):
            print(f"Loaded matching OCR results for {len(cached):,} images")
            return cached
        print("Images or OCR settings changed; refreshing the saved results.")
    reader = create_ocr_reader(model_dir)
    results = extract_image_text(
        dataset, reader, english_words,
        min_english_ratio=MIN_ENGLISH_RATIO, min_confidence=MIN_OCR_CONFIDENCE,
    )
    results["ocr_cache_key"] = cache_key
    results.to_csv(results_path, index=False)
    print(f"Saved OCR results to {results_path}")
    return results



def ocr_attempt_diagnostics(attempt, prefix):
    """Name the same diagnostic fields for an original or retry attempt.

    Args:
        attempt (dict | None): OCR attempt, or None if it was not run.
        prefix (str): Original, flipped, or enhanced column prefix.

    Returns:
        dict: Text, confidence, and vocabulary coverage with prefixed names.
    """
    attempt = attempt or {}
    return {
        f"{prefix}_text": attempt.get("extracted_text", ""),
        f"{prefix}_mean_confidence": attempt.get("mean_confidence", np.nan),
        f"{prefix}_english_word_ratio": attempt.get("english_word_ratio", np.nan),
    }

In [ ]:
def normalize_ocr_text(text):
    """Normalize Unicode, case, apostrophes, and spacing in OCR text.

    Args:
        text (str): Raw OCR text, possibly missing.

    Returns:
        str: Normalized text, or an empty string for missing values.
    """
    if pd.isna(text):
        return ""
    text = unicodedata.normalize("NFKC", str(text)).lower()
    text = text.replace("’", "'").replace("‘", "'")
    return re.sub(r"\s+", " ", text).strip()


def build_cleaning_vocabulary(english_words, alcohol_terms):
    """Combine English words and component words from alcohol terms.

    Args:
        english_words (set[str]): Vocabulary loaded in the OCR section.
        alcohol_terms (set[str]): Imported alcohol-brand and drink terms.

    Returns:
        tuple: Combined vocabulary and candidate words grouped by length.
    """
    vocabulary = set(english_words)
    for term in alcohol_terms:
        vocabulary.update(
            re.findall(r"[^\W_]+(?:'[^\W_]+)*", normalize_ocr_text(term))
        )
    word_groups = defaultdict(list)
    for word in sorted(vocabulary):
        word_groups[len(word)].append(word)
    return vocabulary, word_groups


def calculate_ocr_edit_distance(word, target, confusions):
    """Measure edit distance with smaller costs for common OCR substitutions.

    Args:
        word (str): OCR token.
        target (str): Candidate vocabulary word.
        confusions (dict[str, str]): Characters that are easily confused.

    Returns:
        float: Edit cost; insertions/deletions cost one and confusions cost 0.25.
    """
    previous = list(range(len(target) + 1))
    for row, character in enumerate(word, start=1):
        current = [row]
        for column, expected in enumerate(target, start=1):
            substitution = 0.0 if character == expected else 1.0
            if (
                character != expected
                and (expected in confusions.get(character, "")
                     or character in confusions.get(expected, ""))
            ):
                substitution = 0.25
            current.append(min(
                previous[column] + 1, current[column - 1] + 1,
                previous[column - 1] + substitution,
            ))
        previous = current
    return previous[-1]


def correct_ocr_word(
    word, vocabulary, word_groups, confusions, min_similarity,
    previous_word="", next_word="", phrase_pairs=frozenset(),
):
    """Correct an unknown token only when one OCR-aware match clearly wins.

    Args:
        word (str): Normalized OCR token.
        vocabulary (set[str]): Shared English and beverage vocabulary.
        word_groups (dict): Candidate words grouped by length.
        confusions (dict[str, str]): Possible digit and letter substitutions.
        min_similarity (float): Minimum edit-based similarity.
        previous_word (str): Previous token, used as phrase evidence.
        next_word (str): Next token, used as phrase evidence.
        phrase_pairs (set[tuple]): Known neighboring words in domain phrases.

    Returns:
        str: Reliable correction, or the unchanged token if evidence is weak.
    """
    if word in vocabulary or not any(character.isalpha() for character in word):
        return word
    possible_words = [
        target for length in range(max(1, len(word) - 1), len(word) + 2)
        for target in word_groups.get(length, [])
    ]
    candidates = set(get_close_matches(word, possible_words, n=8, cutoff=0.75))
    candidates.update(
        target for before, target in phrase_pairs if before == previous_word
    )
    candidates.update(
        target for target, after in phrase_pairs if after == next_word
    )
    # Direct substitutions also recover short words that fuzzy matching misses.
    for index, character in enumerate(word):
        alternatives = set(confusions.get(character, ""))
        alternatives.update(
            source for source, values in confusions.items() if character in values
        )
        for alternative in alternatives:
            candidate = word[:index] + alternative + word[index + 1:]
            if candidate in vocabulary:
                candidates.add(candidate)
    scores = []
    for target in candidates:
        distance = calculate_ocr_edit_distance(word, target, confusions)
        similarity = 1 - distance / max(len(word), len(target))
        phrase_match = (
            (previous_word, target) in phrase_pairs
            or (target, next_word) in phrase_pairs
        )
        if phrase_match:
            similarity += 0.15
        scores.append((min(similarity, 1.0), target))
    scores.sort()
    if not scores or scores[-1][0] < min_similarity:
        return word
    if len(scores) > 1 and scores[-1][0] - scores[-2][0] < MIN_CORRECTION_MARGIN:
        return word
    return scores[-1][1]


def split_merged_ocr_word(word, vocabulary):
    """Recover spaces in long tokens while preserving all original characters.

    Args:
        word (str): Normalized OCR token.
        vocabulary (set[str]): Common text and beverage words of length >= 3.

    Returns:
        str: Segmented token when at least three words cover 75% of its letters.
    """
    if len(word) < 16 or word in vocabulary:
        return word
    # Unknown characters cost more than a recognized word, favoring long matches.
    best = [(0.0, [])] + [(float("inf"), []) for _ in word]
    max_length = max(map(len, vocabulary), default=0)
    for end in range(1, len(word) + 1):
        cost, parts = best[end - 1]
        best[end] = (cost + 1, parts + [(word[end - 1], False)])
        for start in range(max(0, end - max_length), end - 2):
            token = word[start:end]
            if token in vocabulary and best[start][0] + 0.5 < best[end][0]:
                best[end] = (best[start][0] + 0.5, best[start][1] + [(token, True)])
    parts = best[-1][1]
    recognized = [token for token, known in parts if known]
    if len(recognized) < 3 or sum(map(len, recognized)) < len(word) * 0.75:
        return word
    grouped = []
    for token, known in parts:
        if not known and grouped and not grouped[-1][1]:
            grouped[-1] = (grouped[-1][0] + token, False)
        else:
            grouped.append((token, known))
    return " ".join(token for token, _ in grouped)


def clean_ocr_results(
    results, english_words, alcohol_terms, number_to_letter, min_similarity=0.90
):
    """Add cleaned OCR text and count each applied token correction.

    Args:
        results (pd.DataFrame): OCR results containing extracted_text.
        english_words (set[str]): Vocabulary loaded once in the OCR section.
        alcohol_terms (set[str]): Additional alcohol-brand and drink terms.
        number_to_letter (dict[str, str]): Possible digit-to-letter substitutions.
        min_similarity (float): Minimum OCR-aware correction similarity.

    Returns:
        tuple: Copy of OCR results with cleaned_text, and a correction-count table.
    """
    vocabulary, word_groups = build_cleaning_vocabulary(english_words, alcohol_terms)
    confusions = {**LETTER_CONFUSIONS, **number_to_letter}
    segmentation_words = {
        word for term in alcohol_terms | BEVERAGE_WORDS | COMMON_TEXT_WORDS
        for word in re.findall(r"[^\W_]+(?:'[^\W_]+)*", normalize_ocr_text(term))
        if len(word) >= 3
    }
    phrase_pairs = set()
    for phrase in alcohol_terms | COMMON_PHRASES:
        tokens = normalize_ocr_text(phrase).split()
        phrase_pairs.update(zip(tokens, tokens[1:]))
    correction_cache = {}
    correction_counts = Counter()

    def replace_word(match):
        """Replace one token while recording every corrected occurrence.

        Args:
            match (re.Match): Token matched within normalized OCR text.

        Returns:
            str: Corrected token or its unchanged normalized form.
        """
        word = match.group()
        before = word_pattern.findall(match.string[:match.start()])
        after = word_pattern.findall(match.string[match.end():])
        previous_word = before[-1] if before else ""
        next_word = after[0] if after else ""
        cache_key = (word, previous_word, next_word)
        if cache_key not in correction_cache:
            corrected = correct_ocr_word(
                word, vocabulary, word_groups, confusions, min_similarity,
                previous_word, next_word, phrase_pairs,
            )
            correction_cache[cache_key] = split_merged_ocr_word(
                corrected, segmentation_words
            )
        corrected = correction_cache[cache_key]
        if corrected != word:
            correction_counts[(word, corrected)] += 1
        return corrected

    cleaned_results = results.copy()
    normalized_texts = results["extracted_text"].map(normalize_ocr_text)
    word_pattern = re.compile(r"[^\W_]+(?:'[^\W_]+)*")
    cleaned_results["cleaned_text"] = normalized_texts.map(
        lambda text: word_pattern.sub(replace_word, text)
    )
    corrections = pd.DataFrame(
        [
            {
                "original_token": original,
                "corrected_token": corrected,
                "occurrences": count,
            }
            for (original, corrected), count in sorted(
                correction_counts.items(), key=lambda item: (-item[1], item[0])
            )
        ],
        columns=["original_token", "corrected_token", "occurrences"],
    )
    return cleaned_results, corrections

In [ ]:
english_words = load_english_vocabulary(BEVERAGE_WORDS)
ocr_results_path = OCR_RESULTS_DIR / "ocr_text.csv"
ocr_results = load_or_extract_ocr(
    dataset, ocr_results_path, CACHE_DIR / "easyocr", english_words
)

In [ ]:
ocr_summary = (
    ocr_results.assign(has_text=ocr_results["word_count"].gt(0))
    .groupby("label")
    .agg(
        image_count=("file_name", "size"),
        images_with_text=("has_text", "sum"),
        mean_word_count=("word_count", "mean"),
        mean_confidence=("mean_confidence", "mean"),
        mean_english_word_ratio=("english_word_ratio", "mean"),
        flip_attempts=("flip_attempted", "sum"),
    )
)
ocr_summary["images_without_text"] = (
    ocr_summary["image_count"] - ocr_summary["images_with_text"]
)
ocr_summary["text_coverage_percent"] = (
    ocr_summary["images_with_text"] / ocr_summary["image_count"] * 100
)
ocr_summary.to_csv(OCR_RESULTS_DIR / "ocr_summary.csv")

print(
    f"Images with extracted text: {ocr_results['word_count'].gt(0).sum():,}"
    f" / {len(ocr_results):,}"
)
display(ocr_summary)
display(
    ocr_results.groupby("label", sort=True)
    .head(3)[
        [
            "file_name",
            "label",
            "extracted_text",
            "word_count",
            "mean_confidence",
        ]
    ]
    .style.format(
        {"word_count": "{:,.0f}", "mean_confidence": "{:,.2f}"}, na_rep="N/A"
    )
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].bar(
    ocr_summary.index,
    ocr_summary["text_coverage_percent"],
    color=["#4878A8", "#59A14F"],
)
axes[0].set(
    title="OCR Text Coverage",
    xlabel="Class",
    ylabel="Images with text (%)",
    ylim=(0, 115),
)
axes[0].yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
for position, coverage in enumerate(ocr_summary["text_coverage_percent"]):
    axes[0].text(
        position,
        coverage,
        f"{coverage:,.2f}%",
        ha="center",
        va="bottom",
        fontsize=18,
    )

axes[1].hist(ocr_results["word_count"], bins=20, color="#4878A8")
axes[1].set(
    title="Extracted Word Counts",
    xlabel="Words per image",
    ylabel="Image count",
)
axes[1].xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
axes[1].yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
save_figure(fig, OCR_RESULTS_DIR / 'ocr_statistics.png')

meets_thresholds = (
    ocr_results["english_word_ratio"].ge(MIN_ENGLISH_RATIO)
    & ocr_results["mean_confidence"].ge(MIN_OCR_CONFIDENCE)
)
retry_summary = pd.DataFrame(
    {
        "images": [len(ocr_results)],
        "flip_attempts": [ocr_results["flip_attempted"].sum()],
        "flipped_selections": [ocr_results["selected_orientation"].eq("flipped").sum()],
        "enhancement_attempts": [ocr_results["enhancement_attempted"].sum()],
        "enhanced_selections": [ocr_results["selected_preprocessing"].ne("native").sum()],
        "remaining_below_threshold": [(~meets_thresholds).sum()],
    }
)
comparison_columns = [
    "file_name", "label", "flip_attempted", "selected_orientation",
    "selected_preprocessing", "quality_score", "enhancement_attempted",
    "enhanced_text", "enhanced_mean_confidence", "enhanced_english_word_ratio",
    "original_text", "original_english_word_ratio", "original_mean_confidence",
    "flipped_text", "flipped_english_word_ratio", "flipped_mean_confidence",
    "extracted_text", "english_word_ratio", "mean_confidence",
]
ocr_comparison = ocr_results[comparison_columns]
retry_summary.to_csv(OCR_RESULTS_DIR / "ocr_retry_summary.csv", index=False)
ocr_comparison.to_csv(OCR_RESULTS_DIR / "ocr_attempt_comparison.csv", index=False)

display(retry_summary.style.format("{:,.0f}"))

fig, axis = plt.subplots(figsize=(12, 6))
for column, label, color in (
    ("original_mean_confidence", "Original", "#4878A8"),
    ("mean_confidence", "Selected", "#59A14F"),
):
    axis.hist(
        ocr_results[column].dropna(), bins=np.linspace(0, 1, 11),
        alpha=0.60, label=label, color=color,
    )
axis.axvline(
    MIN_OCR_CONFIDENCE, linestyle="--", color="black", label="80% threshold"
)
axis.set(
    title="Original and Selected OCR Confidence",
    xlabel="Mean confidence", ylabel="Images",
)
axis.xaxis.set_major_formatter(StrMethodFormatter("{x:,.0%}"))
axis.yaxis.set_major_locator(MaxNLocator(integer=True))
axis.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
axis.legend()

save_figure(fig, OCR_RESULTS_DIR / 'ocr_retry_confidence.png')

## Clean Text

Preserve raw text and add normalized `cleaned_text`.  
Correct unknown words only when an OCR-aware vocabulary match clearly wins.

In [ ]:
ocr_results, cleaning_corrections = clean_ocr_results(
    ocr_results,
    english_words,
    ALCOHOL_TERMS,
    NUMBER_TO_LETTER,
    min_similarity=MIN_WORD_SIMILARITY,
)
ocr_results.to_csv(OCR_RESULTS_DIR / "ocr_text_cleaned.csv", index=False)

In [ ]:
normalized_text = ocr_results["extracted_text"].map(normalize_ocr_text)
changed_text = (
    ocr_results["extracted_text"].fillna("").astype(str)
    != ocr_results["cleaned_text"]
)
cleaning_summary = pd.DataFrame(
    {
        "images": [len(ocr_results)],
        "changed_images": [changed_text.sum()],
        "spelling_or_spacing_changed_images": [
            normalized_text.ne(ocr_results["cleaned_text"]).sum()
        ],
        "corrected_tokens": [cleaning_corrections["occurrences"].sum()],
        "unique_corrections": [len(cleaning_corrections)],
        "empty_cleaned_texts": [ocr_results["cleaned_text"].eq("").sum()],
    }
)
cleaning_summary.to_csv(OCR_RESULTS_DIR / "text_cleaning_summary.csv", index=False)
cleaning_corrections.to_csv(
    OCR_RESULTS_DIR / "text_cleaning_corrections.csv", index=False
)

display(cleaning_summary.style.format("{:,.0f}"))
print("Changed images, including normalization-only changes:")
display(
    ocr_results.loc[
        changed_text, ["file_name", "label", "extracted_text", "cleaned_text"]
    ].head(10)
)

fig, axis = plt.subplots(figsize=(14, 8))
top_corrections = cleaning_corrections.head(15)
if top_corrections.empty:
    axis.text(
        0.5, 0.5, "No token corrections",
        ha="center", va="center", transform=axis.transAxes,
    )
else:
    correction_labels = (
        top_corrections["original_token"] + " → " + top_corrections["corrected_token"]
    )
    axis.barh(correction_labels, top_corrections["occurrences"], color="#4878A8")
    axis.invert_yaxis()
    for position, count in enumerate(top_corrections["occurrences"]):
        axis.text(count, position, f" {count:,}", va="center", fontsize=16)
    axis.set_xlim(0, top_corrections["occurrences"].max() * 1.15)
axis.set(title="Most Frequent OCR Token Corrections", xlabel="Corrected occurrences")
axis.xaxis.set_major_locator(MaxNLocator(integer=True))
axis.xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
save_figure(fig, OCR_RESULTS_DIR / 'text_cleaning_corrections.png')

# Text Embedding

Build five representations from cleaned text, preserving image row order.

## Utils

In [ ]:
def create_tfidf_embeddings(
    texts, analyzer="char_wb", ngram_range=(3, 5), max_features=5000
):
    """Fit word or character TF-IDF features to OCR text.

    Args:
        texts (pd.Series): Cleaned OCR text in image row order.
        analyzer (str): Word or char_wb feature extraction.
        ngram_range (tuple[int, int]): Minimum and maximum n-gram lengths.
        max_features (int): Maximum number of vocabulary features.

    Returns:
        tuple: Sparse feature matrix and fitted vectorizer.

    Raises:
        ValueError: If no usable vocabulary is available.
    """
    vectorizer = TfidfVectorizer(
        analyzer=analyzer, ngram_range=ngram_range, max_features=max_features,
        sublinear_tf=True, norm="l2", dtype=np.float32,
    )
    return vectorizer.fit_transform(texts.tolist()), vectorizer


def summarize_tfidf(embeddings, name):
    """Summarize the size and sparsity of a TF-IDF matrix.

    Args:
        embeddings (scipy.sparse.csr_matrix): Image feature matrix.
        name (str): Label for the summary row.

    Returns:
        pd.DataFrame: Image, feature, nonzero, sparsity, and zero-vector counts.
    """
    return pd.DataFrame({
        "images": [embeddings.shape[0]],
        "features": [embeddings.shape[1]],
        "nonzero_values": [embeddings.nnz],
        "sparsity_percent": [(1 - embeddings.nnz / np.prod(embeddings.shape)) * 100],
        "zero_vectors": [np.count_nonzero(embeddings.getnnz(axis=1) == 0)],
    }, index=[name])


def describe_tfidf_features(embeddings, vectorizer, feature_column):
    """Create one row per fitted word or character feature.

    Args:
        embeddings (scipy.sparse.csr_matrix): Image feature matrix.
        vectorizer (TfidfVectorizer): Fitted feature vocabulary.
        feature_column (str): Column name for words or character fragments.

    Returns:
        pd.DataFrame: Feature indices, names, IDF values, and mean weights.
    """
    return pd.DataFrame({
        "feature_index": np.arange(embeddings.shape[1]),
        feature_column: vectorizer.get_feature_names_out(),
        "idf": vectorizer.idf_,
        "mean_tfidf": np.asarray(embeddings.mean(axis=0)).ravel(),
    })


def plot_tfidf_features(features, feature_column, title, output_path):
    """Plot the fifteen highest-weight words or character fragments.

    Args:
        features (pd.DataFrame): Feature names and mean TF-IDF weights.
        feature_column (str): Column containing feature names.
        title (str): Figure title.
        output_path (Path): Saved figure destination.
    """
    top_features = features.nlargest(15, "mean_tfidf")
    labels = top_features[feature_column]
    if feature_column == "character_fragment":
        labels = labels.map(repr)
    fig, axis = plt.subplots(figsize=(12, 8))
    axis.barh(labels, top_features["mean_tfidf"], color="#4878A8")
    axis.invert_yaxis()
    axis.set(title=title, xlabel="Mean TF-IDF weight")
    axis.xaxis.set_major_formatter(StrMethodFormatter("{x:,.3f}"))
    save_figure(fig, output_path)


def save_pca_results(
    projection, pca, name, output_dir, prefix, point_size=60, alpha=0.65
):
    """Save PCA coordinates and variance, then display the projection.

    Args:
        projection (pd.DataFrame | None): Metadata with pc1/pc2 coordinates.
        pca (PCA | None): Fitted projection model, if available.
        name (str): Embedding name for the plot.
        output_dir (Path): Embedding results directory.
        prefix (str): Prefix for saved filenames.
        point_size (int): Scatter point size.
        alpha (float): Scatter point opacity.
    """
    if projection is None:
        print(f"{name}: PCA needs at least two distinct nonzero vectors.")
        return
    variance = pd.DataFrame({
        "component": ["PC1", "PC2"],
        "explained_variance_percent": pca.explained_variance_ratio_ * 100,
    })
    projection.to_csv(output_dir / f"{prefix}_pca.csv", index=False)
    variance.to_csv(output_dir / f"{prefix}_pca_summary.csv", index=False)
    display_pca_projection(
        projection, pca, name, output_dir / f"{prefix}_pca.png", point_size, alpha
    )

In [ ]:
def display_pca_projection(
    projection, pca, embedding_name, output_path, point_size=60, alpha=0.65
):
    """Save and display a PCA scatter plot colored by image class.

    Args:
        projection (pd.DataFrame): Image labels and pc1/pc2 coordinates.
        pca (PCA): Fitted PCA with explained_variance_ratio_ for axis labels.
        embedding_name (str): Embedding name displayed in the plot title.
        output_path (Path): Figure path in the selected results directory.
        point_size (int): Size of scatter points.
        alpha (float): Opacity of scatter points.
    """
    fig, axis = plt.subplots(figsize=(12, 8))
    for label, color in (("alcohol", "#4878A8"), ("non_alcohol", "#59A14F")):
        points = projection.loc[projection["label"].eq(label)]
        axis.scatter(
            points["pc1"],
            points["pc2"],
            label=label,
            color=color,
            s=point_size,
            alpha=alpha,
        )
    axis.set(
        title=f"{embedding_name} Embeddings: PCA Projection",
        xlabel=f"PC1 ({pca.explained_variance_ratio_[0] * 100:,.2f}% variance)",
        ylabel=f"PC2 ({pca.explained_variance_ratio_[1] * 100:,.2f}% variance)",
    )
    axis.xaxis.set_major_formatter(StrMethodFormatter("{x:,.2f}"))
    axis.yaxis.set_major_formatter(StrMethodFormatter("{x:,.2f}"))
    axis.legend()
    save_figure(fig, output_path)

In [ ]:
def create_tfidf_pca_projection(embeddings, metadata):
    """Project nonzero sparse TF-IDF vectors onto two principal components.

    Args:
        embeddings (scipy.sparse.csr_matrix): TF-IDF vectors in image row order.
        metadata (pd.DataFrame): Corresponding image metadata and labels.

    Returns:
        tuple: Image metadata with pc1 and pc2 coordinates, and the fitted PCA.

    Raises:
        ValueError: If fewer than three nonzero vectors or features are available.
    """
    has_features = embeddings.getnnz(axis=1) > 0
    nonzero_embeddings = embeddings[has_features]
    if min(nonzero_embeddings.shape) <= 2:
        raise ValueError(
            "ARPACK PCA needs at least three nonzero images and features."
        )

    pca = PCA(n_components=2, svd_solver="arpack", random_state=42)
    coordinates = pca.fit_transform(nonzero_embeddings)
    projection = metadata.loc[has_features].copy()
    projection[["pc1", "pc2"]] = coordinates
    return projection, pca

In [ ]:
def create_combined_tfidf_embeddings(word_embeddings, character_embeddings):
    """Concatenate word and character TF-IDF features and normalize each row.

    Args:
        word_embeddings (scipy.sparse.csr_matrix): L2-normalized word features.
        character_embeddings (scipy.sparse.csr_matrix): L2-normalized character
            features in the same image row order.

    Returns:
        scipy.sparse.csr_matrix: Float32 combined features, with word columns
            first, character columns second, and unit norm for nonzero rows.
    """
    combined = sparse.hstack(
        [word_embeddings, character_embeddings], format="csr", dtype=np.float32
    )
    return normalize(combined, norm="l2", copy=False)

In [ ]:
def create_bert_embeddings(texts, model, batch_size=32):
    """Encode nonempty OCR text and preserve zero vectors for empty text.

    Args:
        texts (pd.Series): OCR text with missing values replaced by strings.
        model (SentenceTransformer): Initialized CPU sentence encoder.
        batch_size (int): Number of texts to encode in each batch.

    Returns:
        np.ndarray: Float32 feature matrix in input row order. Nonempty
            vectors have unit L2 norm; empty text has an all-zero vector.
    """
    embeddings = np.zeros(
        (len(texts), model.get_sentence_embedding_dimension()), dtype=np.float32
    )
    has_text = texts.str.strip().ne("").to_numpy()
    if has_text.any():
        embeddings[has_text] = model.encode(
            texts.loc[has_text].tolist(),
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
    return embeddings

In [ ]:
def create_glove_embeddings(tokenized_texts, word_vectors, dimensions=100):
    """Average known GloVe words and normalize one vector per image.

    Args:
        tokenized_texts (pd.Series): Token lists in image row order.
        word_vectors (dict[str, np.ndarray]): Pretrained vectors for known words.
        dimensions (int): Number of features in each pretrained vector.

    Returns:
        tuple: Float32 embedding matrix and per-image token coverage DataFrame.
            Text with no known tokens retains an all-zero vector.
    """
    embeddings = np.zeros((len(tokenized_texts), dimensions), dtype=np.float32)
    token_counts = []
    matched_counts = []
    for row, tokens in enumerate(tokenized_texts):
        known_vectors = [word_vectors[token] for token in tokens if token in word_vectors]
        token_counts.append(len(tokens))
        matched_counts.append(len(known_vectors))
        if known_vectors:
            embeddings[row] = np.mean(known_vectors, axis=0)

    embeddings = normalize(embeddings, norm="l2", copy=False)
    coverage = pd.DataFrame(
        {"token_count": token_counts, "matched_token_count": matched_counts}
    )
    coverage["unknown_token_count"] = (
        coverage["token_count"] - coverage["matched_token_count"]
    )
    coverage["coverage_percent"] = (
        coverage["matched_token_count"]
        .div(coverage["token_count"].replace(0, np.nan))
        .fillna(0) * 100
    )
    coverage["vector_norm"] = np.linalg.norm(embeddings, axis=1)
    return embeddings, coverage

In [ ]:
def create_dense_pca_projection(embeddings, metadata, require_distinct=True):
    """Project nonzero dense vectors onto two principal components.

    Args:
        embeddings (np.ndarray): Dense vectors in image row order.
        metadata (pd.DataFrame): Corresponding image metadata and labels.
        require_distinct (bool): Skip identical vectors for GloVe projections.

    Returns:
        tuple: Metadata with PCA coordinates and fitted PCA, or (None, None)
            if fewer than two nonzero vectors are available, or all vectors
            are identical when require_distinct is True.
    """
    has_vectors = np.linalg.norm(embeddings, axis=1) > 0
    nonzero_embeddings = embeddings[has_vectors]
    if len(nonzero_embeddings) < 2:
        return None, None
    if require_distinct and np.all(nonzero_embeddings == nonzero_embeddings[0]):
        return None, None

    pca = PCA(n_components=2, svd_solver="full")
    coordinates = pca.fit_transform(nonzero_embeddings)
    projection = metadata.loc[has_vectors].copy()
    projection[["pc1", "pc2"]] = coordinates
    return projection, pca

In [ ]:
def compare_embedding_silhouettes(embedding_methods, labels):
    """Compare class separation on a shared subset of nonzero image vectors.

    Args:
        embedding_methods (dict): Method names mapped to (feature matrix,
            model name) pairs. Matrices may be dense or sparse and must use
            the same image row order as labels.
        labels (pd.Series): Alcohol/non-alcohol label for each image.

    Returns:
        tuple: Comparison DataFrame sorted by descending cosine silhouette
            score, and the shared boolean row mask. Undefined scores are NaN
            and sort last.

    Raises:
        ValueError: If a feature matrix and labels have different row counts.
    """
    labels = np.asarray(labels)
    shared_rows = np.ones(len(labels), dtype=bool)
    for embeddings, _ in embedding_methods.values():
        if embeddings.shape[0] != len(labels):
            raise ValueError("All embeddings must have one row per image label.")
        if sparse.issparse(embeddings):
            has_vector = np.asarray(
                embeddings.multiply(embeddings).sum(axis=1)
            ).ravel() > 0
        else:
            has_vector = np.linalg.norm(embeddings, axis=1) > 0
        shared_rows &= has_vector

    comparison_labels = labels[shared_rows]
    group_count = len(np.unique(comparison_labels))
    can_score = 2 <= group_count < len(comparison_labels)
    rows = []
    for method, (embeddings, model) in embedding_methods.items():
        score = (
            silhouette_score(
                embeddings[shared_rows], comparison_labels, metric="cosine"
            )
            if can_score else np.nan
        )
        rows.append(
            {
                "embedding_method": method,
                "model": model,
                "dimensions": embeddings.shape[1],
                "silhouette_score": score,
            }
        )

    comparison = pd.DataFrame(rows).sort_values(
        "silhouette_score", ascending=False, na_position="last", kind="stable"
    ).reset_index(drop=True)
    return comparison, shared_rows

## Preparations

In [ ]:
EMBEDDING_RESULTS_DIR = RESULTS_DIR / "embeddings"
TFIDF_RESULTS_DIR = EMBEDDING_RESULTS_DIR / "tfidf"
WORD_TFIDF_RESULTS_DIR = EMBEDDING_RESULTS_DIR / "word_tfidf"
COMBINED_TFIDF_RESULTS_DIR = EMBEDDING_RESULTS_DIR / "combined_tfidf"
BERT_RESULTS_DIR = EMBEDDING_RESULTS_DIR / "bert"
TFIDF_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
WORD_TFIDF_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
COMBINED_TFIDF_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
BERT_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
embedding_texts = ocr_results["cleaned_text"].reset_index(drop=True)
embedding_metadata = ocr_results[
    ["file_name", "image_path", "label", "extracted_text", "cleaned_text"]
].reset_index(drop=True)
embedding_metadata.insert(0, "embedding_row", np.arange(len(embedding_texts)))
embedding_metadata["has_text"] = embedding_texts.ne("")

embedding_metadata.to_csv(
    EMBEDDING_RESULTS_DIR / "embedding_metadata.csv", index=False
)

## Word TF-IDF

Use word unigrams and bigrams, up to 5,000 features, sublinear term frequency,
and L2 normalization. Keep stop words and numbers.

In [ ]:
word_tfidf_embeddings, word_tfidf_vectorizer = create_tfidf_embeddings(
    embedding_texts, analyzer="word", ngram_range=(1, 2)
)
sparse.save_npz(
    WORD_TFIDF_RESULTS_DIR / "word_tfidf_embeddings.npz", word_tfidf_embeddings
)
joblib.dump(
    word_tfidf_vectorizer, WORD_TFIDF_RESULTS_DIR / "word_tfidf_vectorizer.joblib"
)

In [ ]:
word_tfidf_summary = summarize_tfidf(word_tfidf_embeddings, "Word TF-IDF")
word_tfidf_features = describe_tfidf_features(
    word_tfidf_embeddings, word_tfidf_vectorizer, "word_or_phrase"
)
word_tfidf_summary.to_csv(WORD_TFIDF_RESULTS_DIR / "word_tfidf_summary.csv", index=False)
word_tfidf_features.to_csv(WORD_TFIDF_RESULTS_DIR / "word_tfidf_features.csv", index=False)
display(word_tfidf_summary)
plot_tfidf_features(
    word_tfidf_features, "word_or_phrase", "Top Words and Phrases",
    WORD_TFIDF_RESULTS_DIR / "word_tfidf_statistics.png",
)

In [ ]:
word_tfidf_projection, word_tfidf_pca = create_tfidf_pca_projection(
    word_tfidf_embeddings, embedding_metadata
)

In [ ]:
save_pca_results(
    word_tfidf_projection, word_tfidf_pca, "Word TF-IDF",
    WORD_TFIDF_RESULTS_DIR, "word_tfidf",
)

## Character TF-IDF

Use 3–5-character fragments within word boundaries to capture partial OCR
words. Use the same feature limit and normalization as word TF-IDF.

In [ ]:
tfidf_embeddings, tfidf_vectorizer = create_tfidf_embeddings(embedding_texts)
sparse.save_npz(
    TFIDF_RESULTS_DIR / "tfidf_embeddings.npz", tfidf_embeddings
)
joblib.dump(tfidf_vectorizer, TFIDF_RESULTS_DIR / "tfidf_vectorizer.joblib")

In [ ]:
tfidf_summary = summarize_tfidf(tfidf_embeddings, "TF-IDF")
tfidf_features = describe_tfidf_features(
    tfidf_embeddings, tfidf_vectorizer, "character_fragment"
)
tfidf_summary.to_csv(TFIDF_RESULTS_DIR / "tfidf_summary.csv", index=False)
tfidf_features.to_csv(TFIDF_RESULTS_DIR / "tfidf_features.csv", index=False)
display(tfidf_summary)
plot_tfidf_features(
    tfidf_features, "character_fragment", "Top Character Fragments",
    TFIDF_RESULTS_DIR / "tfidf_statistics.png",
)

In [ ]:
tfidf_projection, tfidf_pca = create_tfidf_pca_projection(
    tfidf_embeddings, embedding_metadata
)

In [ ]:
save_pca_results(
    tfidf_projection, tfidf_pca, "TF-IDF",
    TFIDF_RESULTS_DIR, "tfidf",
)

## Word + Character TF-IDF

Concatenate word features first, then character features, and L2-normalize
each row. Empty text retains a zero vector.

In [ ]:
combined_tfidf_embeddings = create_combined_tfidf_embeddings(
    word_tfidf_embeddings, tfidf_embeddings
)
sparse.save_npz(
    COMBINED_TFIDF_RESULTS_DIR / "combined_tfidf_embeddings.npz",
    combined_tfidf_embeddings,
)
combined_tfidf_vectorizers = {
    "word": word_tfidf_vectorizer,
    "character": tfidf_vectorizer,
}
joblib.dump(
    combined_tfidf_vectorizers,
    COMBINED_TFIDF_RESULTS_DIR / "combined_tfidf_vectorizers.joblib",
)

In [ ]:
combined_tfidf_summary = summarize_tfidf(
    combined_tfidf_embeddings, "Word + Character TF-IDF"
)
combined_tfidf_summary["word_features"] = word_tfidf_embeddings.shape[1]
combined_tfidf_summary["character_features"] = tfidf_embeddings.shape[1]
combined_tfidf_features = pd.concat([
    word_tfidf_features.rename(columns={"word_or_phrase": "feature"})
    .assign(feature_type="word"),
    tfidf_features.rename(columns={"character_fragment": "feature"})
    .assign(feature_type="character"),
], ignore_index=True)
combined_tfidf_features["feature_index"] = np.arange(
    combined_tfidf_embeddings.shape[1]
)
combined_tfidf_features["mean_tfidf"] = np.asarray(
    combined_tfidf_embeddings.mean(axis=0)
).ravel()
combined_tfidf_summary.to_csv(
    COMBINED_TFIDF_RESULTS_DIR / "combined_tfidf_summary.csv", index=False
)
combined_tfidf_features.to_csv(
    COMBINED_TFIDF_RESULTS_DIR / "combined_tfidf_features.csv", index=False
)
display(combined_tfidf_summary)

In [ ]:
combined_tfidf_projection, combined_tfidf_pca = create_tfidf_pca_projection(
    combined_tfidf_embeddings, embedding_metadata
)

In [ ]:
save_pca_results(
    combined_tfidf_projection, combined_tfidf_pca, "Word + Character TF-IDF",
    COMBINED_TFIDF_RESULTS_DIR, "combined_tfidf",
)

## BERT

Encode text with pretrained `all-MiniLM-L6-v2` on the CPU. Normalize the
384-dimensional vectors and keep empty text as zero vectors. Report texts
that exceed the model's token limit.

In [ ]:
bert_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    revision="1110a243fdf4706b3f48f1d95db1a4f5529b4d41",
    device="cpu",
    cache_folder=str(CACHE_DIR / "sentence_transformers"),
)
bert_embeddings = create_bert_embeddings(embedding_texts, bert_model)
np.save(BERT_RESULTS_DIR / "bert_embeddings.npy", bert_embeddings)

In [ ]:
bert_metadata = embedding_metadata.copy()
bert_metadata["token_count"] = bert_model.tokenizer(
    embedding_texts.tolist(), truncation=False, return_length=True
)["length"]
bert_metadata["is_truncated"] = (
    bert_metadata["token_count"] > bert_model.max_seq_length
)
bert_metadata["vector_norm"] = np.linalg.norm(bert_embeddings, axis=1)
bert_summary = pd.DataFrame(
    {
        "images": [bert_embeddings.shape[0]],
        "dimensions": [bert_embeddings.shape[1]],
        "zero_vectors": [np.count_nonzero(bert_metadata["vector_norm"] == 0)],
        "max_tokens": [bert_model.max_seq_length],
        "truncated_texts": [bert_metadata["is_truncated"].sum()],
    },
    index=["MiniLM"],
)
bert_class_summary = bert_metadata.groupby("label").agg(
    image_count=("file_name", "size"),
    images_with_text=("has_text", "sum"),
    mean_token_count=("token_count", "mean"),
    truncated_texts=("is_truncated", "sum"),
)

display(bert_summary)
display(bert_class_summary)
bert_summary.to_csv(BERT_RESULTS_DIR / "bert_summary.csv", index=False)
bert_class_summary.to_csv(BERT_RESULTS_DIR / "bert_class_summary.csv")
bert_metadata.to_csv(BERT_RESULTS_DIR / "bert_metadata.csv", index=False)

In [ ]:
bert_projection, bert_pca = create_dense_pca_projection(
    bert_embeddings, embedding_metadata, require_distinct=False
)

In [ ]:
save_pca_results(
    bert_projection, bert_pca, "MiniLM", BERT_RESULTS_DIR, "bert",
    point_size=80, alpha=0.8,
)

## GloVe

Average known `glove-wiki-gigaword-100` word vectors and L2-normalize each
image vector. Unknown words are skipped; coverage measures how much text
contributes. Averaging loses word order. Respect `GENSIM_DATA_DIR` if set.

In [ ]:
GLOVE_RESULTS_DIR = EMBEDDING_RESULTS_DIR / "glove"
GLOVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Gensim reads its download cache location when the downloader is imported.
os.environ.setdefault("GENSIM_DATA_DIR", str((CACHE_DIR / "glove").resolve()))
import gensim.downloader as api

glove = api.load("glove-wiki-gigaword-100")
GLOVE_DIMENSIONS = glove.vector_size

glove_tokens = (
    embedding_texts.fillna("").astype(str).str.lower()
    .str.findall(r"[a-z]+(?:'[a-z]+)*|[0-9]+")
)
glove_vocabulary = {token for tokens in glove_tokens for token in tokens}
glove_word_vectors = {token: glove[token] for token in glove_vocabulary if token in glove}
glove_embeddings, glove_coverage = create_glove_embeddings(
    glove_tokens, glove_word_vectors, GLOVE_DIMENSIONS
)
glove_metadata = pd.concat([embedding_metadata, glove_coverage], axis=1)
np.save(GLOVE_RESULTS_DIR / "glove_embeddings.npy", glove_embeddings)
glove_metadata.to_csv(GLOVE_RESULTS_DIR / "glove_metadata.csv", index=False)

In [ ]:
glove_token_total = int(glove_metadata["token_count"].sum())
glove_matched_total = int(glove_metadata["matched_token_count"].sum())
glove_summary = pd.DataFrame(
    {
        "images": [len(glove_embeddings)],
        "dimensions": [GLOVE_DIMENSIONS],
        "unique_tokens": [len(glove_vocabulary)],
        "known_unique_tokens": [len(glove_word_vectors)],
        "total_tokens": [glove_token_total],
        "matched_tokens": [glove_matched_total],
        "token_coverage_percent": [
            100 * glove_matched_total / glove_token_total if glove_token_total else 0
        ],
        "zero_vectors": [glove_metadata["vector_norm"].eq(0).sum()],
    },
    index=["GloVe 6B"],
)
glove_class_summary = glove_metadata.groupby("label").agg(
    image_count=("file_name", "size"),
    images_with_text=("has_text", "sum"),
    images_with_vectors=("vector_norm", lambda norms: norms.gt(0).sum()),
    total_tokens=("token_count", "sum"),
    matched_tokens=("matched_token_count", "sum"),
    mean_coverage_percent=("coverage_percent", "mean"),
)
glove_class_summary["token_coverage_percent"] = (
    glove_class_summary["matched_tokens"]
    .div(glove_class_summary["total_tokens"].replace(0, np.nan))
    .fillna(0) * 100
)
glove_token_counts = Counter(token for tokens in glove_tokens for token in tokens)
glove_vocabulary_table = pd.DataFrame(
    [
        {"token": token, "count": count, "in_glove": token in glove_word_vectors}
        for token, count in glove_token_counts.most_common()
    ],
    columns=["token", "count", "in_glove"],
)

display(glove_summary)
display(glove_class_summary)
print("Most frequent tokens missing from GloVe:")
display(glove_vocabulary_table.loc[~glove_vocabulary_table["in_glove"]].head(20))
glove_summary.to_csv(GLOVE_RESULTS_DIR / "glove_summary.csv", index=False)
glove_class_summary.to_csv(GLOVE_RESULTS_DIR / "glove_class_summary.csv")
glove_vocabulary_table.to_csv(GLOVE_RESULTS_DIR / "glove_vocabulary.csv", index=False)

In [ ]:
glove_projection, glove_pca = create_dense_pca_projection(
    glove_embeddings, glove_metadata
)

In [ ]:
save_pca_results(
    glove_projection, glove_pca, "GloVe",
    GLOVE_RESULTS_DIR, "glove",
)

## Compare Embeddings

Compare cosine silhouette scores on the same images, excluding zero vectors
in any method. Higher scores indicate clearer class separation. These are
exploratory scores, not held-out performance.

In [ ]:
embedding_methods = {
    "Word TF-IDF": (word_tfidf_embeddings, "N/A"),
    "Character TF-IDF": (tfidf_embeddings, "N/A"),
    "Word + Character TF-IDF": (combined_tfidf_embeddings, "N/A"),
    "BERT-based": (bert_embeddings, "sentence-transformers/all-MiniLM-L6-v2"),
    "GloVe": (glove_embeddings, "glove-wiki-gigaword-100"),
}
embedding_comparison, embedding_comparison_rows = compare_embedding_silhouettes(
    embedding_methods, embedding_metadata["label"]
)

In [ ]:
print(f"Images compared across all methods: {embedding_comparison_rows.sum():,}")
print(f"Images excluded (zero vector): {(~embedding_comparison_rows).sum():,}")
if embedding_comparison["silhouette_score"].isna().all():
    print("Silhouette scores need between 2 and n_images - 1 groups.")

display(
    embedding_comparison.style.format(
        {"dimensions": "{:,}", "silhouette_score": "{:,.4f}"},
        na_rep="N/A",
    )
)
embedding_comparison.to_csv(
    EMBEDDING_RESULTS_DIR / "embedding_comparison.csv", index=False
)

# Classification

Compare four classifiers across five embeddings on one stratified 70/30 train/test split. 
Alcohol is the positive class. 
Fit TF-IDF on training text only and reuse pretrained MiniLM and GloVe vectors.

Report accuracy, precision, recall, F1, macro F1, ROC AUC, and PR AUC. 

## Utils

In [ ]:
def fit_tfidf_splits(train_texts, test_texts, vectorizers):
    """Fit word and character vocabularies on training text and combine features.

    Args:
        train_texts (pd.Series): Text used to fit vocabularies.
        test_texts (pd.Series): Text to transform with the fitted vocabularies.
        vectorizers (dict): Character and word vectorizer templates.

    Returns:
        tuple: Three feature splits and the two fitted vectorizers.
    """
    feature_splits = {}
    fitted_vectorizers = {}
    for name, template in vectorizers.items():
        vectorizer = clone(template)
        train_features = vectorizer.fit_transform(train_texts)
        test_features = vectorizer.transform(test_texts)
        feature_splits[name] = (train_features, test_features)
        fitted_vectorizers[name] = vectorizer
    feature_splits["combined_tfidf"] = tuple(
        create_combined_tfidf_embeddings(
            feature_splits["word_tfidf"][index], feature_splits["tfidf"][index]
        )
        for index in (0, 1)
    )
    return feature_splits, fitted_vectorizers


def display_metrics(table):
    """Display evaluation metrics with consistent numeric formatting.

    Args:
        table (pd.DataFrame): Held-out or cross-validation metrics.
    """
    formats = {column: "{:,.3f}" for column in table.select_dtypes(include="number")}
    for column in ("train_images", "test_images", "cv_folds", "fold"):
        if column in table:
            formats[column] = "{:,.0f}"
    display(table.style.format(formats, na_rep="N/A"))


EMBEDDING_LABELS = {
    "tfidf": "Character TF-IDF",
    "word_tfidf": "Word TF-IDF",
    "combined_tfidf": "Word + Character TF-IDF",
    "bert": "MiniLM",
    "glove": "GloVe",
}
EMBEDDING_COLORS = {
    "tfidf": "#4878A8",
    "word_tfidf": "#E15759",
    "combined_tfidf": "#B07AA1",
    "bert": "#59A14F",
    "glove": "#F28E2B",
}


def prepare_classification_data(
    texts,
    metadata,
    bert_embeddings,
    glove_embeddings,
    tfidf_vectorizer,
    word_tfidf_vectorizer,
    test_size=0.3,
    random_state=42,
):
    """Split pretrained vectors and fit TF-IDF features on training text only.

    Args:
        texts (pd.Series): Cleaned OCR text in embedding row order.
        metadata (pd.DataFrame): Image metadata including folder labels.
        bert_embeddings (np.ndarray): Pretrained sentence vectors in row order.
        glove_embeddings (np.ndarray): Averaged pretrained word vectors in row order.
        tfidf_vectorizer (TfidfVectorizer): Template for character TF-IDF settings.
        word_tfidf_vectorizer (TfidfVectorizer): Template for word TF-IDF settings.
        test_size (float): Fraction of images held out for evaluation.
        random_state (int): Seed for a reproducible stratified split.

    Returns:
        dict: Feature splits, training and test labels, test image metadata,
            complete split metadata, raw training text for cross-validation,
            and both training-fitted vectorizers.
    """
    metadata = metadata.reset_index(drop=True)
    labels = metadata["label"].map({"non_alcohol": 0, "alcohol": 1}).to_numpy()
    train_indices, test_indices = train_test_split(
        np.arange(len(labels)),
        test_size=test_size,
        stratify=labels,
        random_state=random_state,
    )

    feature_splits, fitted_vectorizers = fit_tfidf_splits(
        texts.iloc[train_indices], texts.iloc[test_indices],
        {"tfidf": tfidf_vectorizer, "word_tfidf": word_tfidf_vectorizer},
    )
    for name, embeddings in (("bert", bert_embeddings), ("glove", glove_embeddings)):
        feature_splits[name] = (embeddings[train_indices], embeddings[test_indices])
    split_metadata = metadata.copy()
    split_metadata["split"] = "train"
    split_metadata.loc[test_indices, "split"] = "test"

    return {
        "feature_splits": feature_splits,
        "train_texts": texts.iloc[train_indices].reset_index(drop=True),
        "y_train": labels[train_indices],
        "y_test": labels[test_indices],
        "test_metadata": metadata.iloc[test_indices].reset_index(drop=True),
        "split_metadata": split_metadata,
        "tfidf_vectorizer": fitted_vectorizers["tfidf"],
        "word_tfidf_vectorizer": fitted_vectorizers["word_tfidf"],
    }


def evaluate_embeddings(classifier, classification_data):
    """Fit and evaluate one classifier per supplied embedding representation.

    Args:
        classifier: Unfitted classifier supporting predict_proba.
        classification_data (dict): Features and labels from the shared split.

    Returns:
        dict: Per-embedding models, classification reports, metrics, held-out
            predictions, alcohol probabilities, and ROC and PR curve points.
    """
    results = {}
    y_train = classification_data["y_train"]
    y_test = classification_data["y_test"]
    for embedding_name, (x_train, x_test) in classification_data[
        "feature_splits"
    ].items():
        model = clone(classifier)
        model.fit(x_train, y_train)
        predictions = model.predict(x_test)
        positive_column = np.flatnonzero(model.classes_ == 1)[0]
        probabilities = model.predict_proba(x_test)[:, positive_column]
        report_values = classification_report(
            y_test,
            predictions,
            labels=[0, 1],
            target_names=["non_alcohol", "alcohol"],
            output_dict=True,
            zero_division=0,
        )
        accuracy = report_values.pop("accuracy")
        report = pd.DataFrame(report_values).T
        report.index.name = "class"
        fpr, tpr, _ = roc_curve(y_test, probabilities, pos_label=1)
        precision, recall, _ = precision_recall_curve(
            y_test, probabilities, pos_label=1
        )
        results[embedding_name] = {
            "model": model,
            "report": report,
            "metrics": {
                "train_images": len(y_train),
                "test_images": len(y_test),
                "accuracy": accuracy,
                "precision": report_values["alcohol"]["precision"],
                "recall": report_values["alcohol"]["recall"],
                "f1": report_values["alcohol"]["f1-score"],
                "macro_f1": report_values["macro avg"]["f1-score"],
                "roc_auc": roc_auc_score(y_test, probabilities),
                "pr_auc": auc(recall, precision),
            },
            "predictions": predictions,
            "probabilities": probabilities,
            "roc_curve": pd.DataFrame({"fpr": fpr, "tpr": tpr}),
            "pr_curve": pd.DataFrame(
                {"recall": recall, "precision": precision}
            ),
        }
    return results


def show_classifier_results(
    classifier_name, results, classification_data, output_dir
):
    """Save evaluation artifacts and display ROC and precision–recall curves.

    Args:
        classifier_name (str): Name displayed in tables and plots.
        results (dict): Per-embedding output from evaluate_embeddings.
        classification_data (dict): Shared split with held-out image metadata.
        output_dir (Path): Classifier results directory.

    Returns:
        pd.DataFrame: Comparison metrics for the classifier's five embeddings.
    """
    summary_rows = []
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    for embedding_name, result in results.items():
        embedding_dir = output_dir / embedding_name
        embedding_dir.mkdir(parents=True, exist_ok=True)
        metrics = result["metrics"]
        summary_rows.append(
            {
                "classifier": classifier_name,
                "embedding": embedding_name,
                **metrics,
            }
        )
        embedding_label = EMBEDDING_LABELS[embedding_name]
        result["report"].to_csv(embedding_dir / "classification_report.csv")
        pd.DataFrame([metrics]).to_csv(
            embedding_dir / "metrics.csv", index=False
        )
        result["roc_curve"].to_csv(embedding_dir / "roc_curve.csv", index=False)
        result["pr_curve"].to_csv(embedding_dir / "pr_curve.csv", index=False)
        joblib.dump(result["model"], embedding_dir / "classifier.joblib")

        predictions = classification_data["test_metadata"].copy()
        predictions["prediction"] = np.where(
            result["predictions"] == 1, "alcohol", "non_alcohol"
        )
        predictions["alcohol_probability"] = result["probabilities"]
        predictions["is_correct"] = predictions["prediction"].eq(
            predictions["label"]
        )
        predictions.to_csv(embedding_dir / "test_predictions.csv", index=False)

        roc_points = result["roc_curve"]
        pr_points = result["pr_curve"]
        axes[0].plot(
            roc_points["fpr"],
            roc_points["tpr"],
            linewidth=2.5,
            label=f"{embedding_label} (AUC={metrics['roc_auc']:,.3f})",
            color=EMBEDDING_COLORS[embedding_name],
        )
        axes[1].plot(
            pr_points["recall"],
            pr_points["precision"],
            linewidth=2.5,
            label=f"{embedding_label} (AUC={metrics['pr_auc']:,.3f})",
            color=EMBEDDING_COLORS[embedding_name],
        )

    axes[0].plot([0, 1], [0, 1], "--", color="gray", label="Chance")
    prevalence = classification_data["y_test"].mean()
    axes[1].axhline(
        prevalence,
        linestyle="--",
        color="gray",
        label=f"Prevalence ({prevalence:,.2f})",
    )
    axes[0].set(
        title=f"{classifier_name}: ROC",
        xlabel="False positive rate",
        ylabel="True positive rate",
    )
    axes[1].set(
        title=f"{classifier_name}: Precision–Recall",
        xlabel="Recall",
        ylabel="Precision",
    )
    for axis in axes:
        axis.set(xlim=(0, 1), ylim=(0, 1.05))
        axis.xaxis.set_major_formatter(StrMethodFormatter("{x:,.2f}"))
        axis.yaxis.set_major_formatter(StrMethodFormatter("{x:,.2f}"))
        axis.legend(fontsize=13, loc="lower right")
    save_figure(fig, output_dir / 'roc_pr_curves.png')
    summary = pd.DataFrame(summary_rows)
    summary.to_csv(output_dir / "metrics_summary.csv", index=False)
    return summary

In [ ]:
def run_logistic_regression(classification_data):
    """Train and evaluate Logistic Regression once per embedding type.

    Args:
        classification_data (dict): Shared feature splits and labels.

    Returns:
        dict: Character, word, combined TF-IDF, MiniLM, and GloVe evaluation results.
    """
    classifier = LogisticRegression(max_iter=1000, random_state=42)
    return evaluate_embeddings(classifier, classification_data)

In [ ]:
def run_xgboost(classification_data):
    """Train and evaluate CPU XGBoost once per embedding type.

    Args:
        classification_data (dict): Shared feature splits and labels.

    Returns:
        dict: Character, word, combined TF-IDF, MiniLM, and GloVe evaluation results.
    """
    classifier = XGBClassifier(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        device="cpu",
        n_jobs=4,
        random_state=42,
    )
    return evaluate_embeddings(classifier, classification_data)

In [ ]:
def run_random_forest(classification_data):
    """Train and evaluate Random Forest once per embedding type.

    Args:
        classification_data (dict): Shared feature splits and labels.

    Returns:
        dict: Character, word, combined TF-IDF, MiniLM, and GloVe evaluation results.
    """
    classifier = RandomForestClassifier(
        n_estimators=200, min_samples_leaf=2, n_jobs=4, random_state=42
    )
    return evaluate_embeddings(classifier, classification_data)

In [ ]:
def run_mlp(classification_data):
    """Estimate MLP performance with five folds and evaluate final models.

    Args:
        classification_data (dict): Shared splits, raw training text, labels,
            and TF-IDF vectorizer templates from prepare_classification_data.

    Returns:
        tuple: Held-out results for five embeddings, per-fold CV scores, and
            per-embedding CV mean/std summary. Held-out result metrics also
            include the CV summary for the final performance comparison.

    Raises:
        ValueError: If either training class has fewer than five images.
    """
    y_train = classification_data["y_train"]
    if np.bincount(y_train, minlength=2).min() < 5:
        raise ValueError("Five-fold MLP CV needs at least five training images per class.")

    classifier = MLPClassifier(
        hidden_layer_sizes=(64,),
        activation="relu",
        solver="adam",
        alpha=0.01,
        max_iter=500,
        tol=0.001,
        n_iter_no_change=20,
        random_state=42,
    )
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    training_texts = classification_data["train_texts"]
    fold_rows = []
    for fold, (fit_indices, validation_indices) in enumerate(
        cv.split(training_texts, y_train), start=1
    ):
        fold_feature_splits, _ = fit_tfidf_splits(
            training_texts.iloc[fit_indices], training_texts.iloc[validation_indices],
            {name: classification_data[f"{name}_vectorizer"]
             for name in ("tfidf", "word_tfidf")},
        )
        for name in ("bert", "glove"):
            training_features = classification_data["feature_splits"][name][0]
            fold_feature_splits[name] = (
                training_features[fit_indices], training_features[validation_indices]
            )
        fold_data = {
            "feature_splits": fold_feature_splits,
            "y_train": y_train[fit_indices],
            "y_test": y_train[validation_indices],
        }
        fold_results = evaluate_embeddings(classifier, fold_data)
        for name, result in fold_results.items():
            fold_rows.append({"embedding": name, "fold": fold, **result["metrics"]})

    fold_scores = pd.DataFrame(fold_rows)
    metric_names = ["accuracy", "precision", "recall", "f1", "macro_f1", "roc_auc", "pr_auc"]
    cv_summary = fold_scores.groupby("embedding", sort=False)[metric_names].agg(
        ["mean", "std"]
    )
    cv_summary.columns = [
        f"cv_{metric}_{statistic}" for metric, statistic in cv_summary.columns
    ]
    cv_summary["cv_folds"] = 5
    cv_summary = cv_summary.reset_index()

    results = evaluate_embeddings(classifier, classification_data)
    cv_metrics_by_embedding = cv_summary.set_index("embedding")
    for name, result in results.items():
        cv_metrics = cv_metrics_by_embedding.loc[name].to_dict()
        result["metrics"].update(cv_metrics)
    return results, fold_scores, cv_summary

In [ ]:
def compare_classifiers(summaries):
    """Combine classifier summaries into one comparison table.

    Args:
        summaries (list[pd.DataFrame]): Per-classifier embedding summaries.

    Returns:
        pd.DataFrame: All classifier and embedding metrics, sorted by PR AUC
            and then ROC AUC for display.
    """
    return (
        pd.concat(summaries, ignore_index=True)
        .sort_values(["pr_auc", "roc_auc"], ascending=False)
        .reset_index(drop=True)
    )

In [ ]:
def save_best_classifier_results(
    comparison, classifier_results, feature_matrices, metadata, output_path
):
    """Save predictions for every image using the highest-ranked trained model.

    Args:
        comparison (pd.DataFrame): Held-out metrics for each classifier and embedding.
        classifier_results (dict): Classifier names mapped to evaluation results.
        feature_matrices (dict): Full-dataset features using training-fitted TF-IDF
            vectorizers or the original pretrained semantic embeddings.
        metadata (pd.DataFrame): Image filenames in feature row order.
        output_path (Path): Destination for the two-column prediction CSV.

    Returns:
        tuple: Selected comparison row and the exported prediction DataFrame.
    """
    best_run = comparison.sort_values(
        ["pr_auc", "roc_auc"], ascending=False, kind="stable"
    ).iloc[0]
    embedding = best_run["embedding"]
    model = classifier_results[best_run["classifier"]][embedding]["model"]
    predictions = model.predict(feature_matrices[embedding])
    results = metadata[["file_name"]].reset_index(drop=True).copy()
    results["prediction"] = np.where(predictions == 1, "alcohol", "non_alcohol")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    results.to_csv(output_path, index=False)
    return best_run, results


## Preparations

In [ ]:
CLASSIFICATION_RESULTS_DIR = RESULTS_DIR / "classification"
CLASSIFICATION_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
classification_data = prepare_classification_data(
    embedding_texts, embedding_metadata, bert_embeddings, glove_embeddings,
    tfidf_vectorizer, word_tfidf_vectorizer,
)
classification_data["split_metadata"].to_csv(
    CLASSIFICATION_RESULTS_DIR / "data_split.csv", index=False
)
joblib.dump(
    classification_data["tfidf_vectorizer"],
    CLASSIFICATION_RESULTS_DIR / "tfidf_training_vectorizer.joblib",
)
joblib.dump(
    classification_data["word_tfidf_vectorizer"],
    CLASSIFICATION_RESULTS_DIR / "word_tfidf_training_vectorizer.joblib",
)

joblib.dump(
    {
        "word": classification_data["word_tfidf_vectorizer"],
        "character": classification_data["tfidf_vectorizer"],
    },
    CLASSIFICATION_RESULTS_DIR / "combined_tfidf_training_vectorizers.joblib",
)

In [ ]:
split_summary = (
    classification_data["split_metadata"]
    .groupby(["split", "label"])
    .size()
    .unstack(fill_value=0)
)
print(f"Training images: {len(classification_data['y_train']):,}")
print(f"Held-out images: {len(classification_data['y_test']):,}")
if test_mode:
    print("Test mode: these metrics are workflow checks on a small dataset.")
display(split_summary)
split_summary.to_csv(CLASSIFICATION_RESULTS_DIR / "split_summary.csv")

## Logistic Regression

Linear baseline with up to 1,000 iterations.

In [ ]:
logistic_results = run_logistic_regression(classification_data)

In [ ]:
logistic_summary = show_classifier_results(
    "Logistic Regression",
    logistic_results,
    classification_data,
    CLASSIFICATION_RESULTS_DIR / "logistic_regression",
)

## XGBoost

Use 200 depth-3 trees, learning rate 0.05, and CPU histogram training.

In [ ]:
xgboost_results = run_xgboost(classification_data)

In [ ]:
xgboost_summary = show_classifier_results(
    "XGBoost",
    xgboost_results,
    classification_data,
    CLASSIFICATION_RESULTS_DIR / "xgboost",
)

## Random Forest

Use 200 trees and at least two training images per leaf.

In [ ]:
random_forest_results = run_random_forest(classification_data)

In [ ]:
random_forest_summary = show_classifier_results(
    "Random Forest",
    random_forest_results,
    classification_data,
    CLASSIFICATION_RESULTS_DIR / "random_forest",
)

## MLPClassifier

Use one 64-unit ReLU layer, Adam, and L2 regularization. Estimate performance
with five stratified folds within the training set, refitting TF-IDF in each
fold. Each training class needs at least five images. Evaluate final models
on the untouched held-out set.

In [ ]:
mlp_results, mlp_cv_fold_scores, mlp_cv_summary = run_mlp(classification_data)

In [ ]:
mlp_summary = show_classifier_results(
    "MLPClassifier",
    mlp_results,
    classification_data,
    CLASSIFICATION_RESULTS_DIR / "mlp",
)
display_metrics(mlp_cv_summary)
mlp_cv_fold_scores.to_csv(
    CLASSIFICATION_RESULTS_DIR / "mlp" / "cv_fold_scores.csv", index=False
)
mlp_cv_summary.to_csv(
    CLASSIFICATION_RESULTS_DIR / "mlp" / "cv_summary.csv", index=False
)

## Performance Comparison

Compare all twenty held-out runs. MLP rows also include training-only CV
means and standard deviations. Per-class reports and predictions are saved
in each classifier's embedding folder.

In [ ]:
performance_comparison = compare_classifiers(
    [logistic_summary, xgboost_summary, random_forest_summary, mlp_summary]
)

In [ ]:
display_metrics(performance_comparison[[
    "classifier", "embedding", "accuracy", "precision", "recall", "f1",
    "macro_f1", "roc_auc", "pr_auc",
]])
performance_comparison.to_csv(
    CLASSIFICATION_RESULTS_DIR / "performance_comparison.csv", index=False
)

fig, axes = plt.subplots(1, 2, figsize=(22, 8))
for axis, metric, title in zip(
    axes, ["roc_auc", "pr_auc"], ["ROC AUC", "PR AUC"]
):
    comparison_plot = performance_comparison.pivot(
        index="classifier", columns="embedding", values=metric
    ).reindex(
        index=["Logistic Regression", "XGBoost", "Random Forest", "MLPClassifier"],
        columns=list(EMBEDDING_LABELS),
    )
    comparison_plot = comparison_plot.rename(columns=EMBEDDING_LABELS)
    comparison_plot.plot.bar(
        ax=axis, rot=15, width=0.85, color=list(EMBEDDING_COLORS.values())
    )
    axis.set(
        title=title,
        xlabel="Classifier",
        ylabel="Area under curve",
        ylim=(0, 1.1),
    )
    axis.yaxis.set_major_formatter(StrMethodFormatter("{x:,.2f}"))
    axis.legend(title="Embedding", fontsize=13)
    for container in axis.containers:
        axis.bar_label(container, fmt="{:,.3f}", padding=3, fontsize=13)
save_figure(fig, CLASSIFICATION_RESULTS_DIR / 'performance_comparison.png')

# Save Results

Select the classifier and embedding with the highest held-out PR AUC, breaking
ties by ROC AUC. Save predictions for every image to `results/results.csv` with
the same `file_name,prediction` columns as `results/example_results.csv`.
Use the selected trained model without refitting it; transform TF-IDF with its
training-fitted vocabulary. In test mode, save test-dataset predictions to `test/results_test/results.csv`.


In [ ]:
classifier_results = {
    "Logistic Regression": logistic_results,
    "XGBoost": xgboost_results,
    "Random Forest": random_forest_results,
    "MLPClassifier": mlp_results,
}
full_tfidf = classification_data["tfidf_vectorizer"].transform(embedding_texts)
full_word_tfidf = classification_data["word_tfidf_vectorizer"].transform(embedding_texts)
full_features = {
    "tfidf": full_tfidf,
    "word_tfidf": full_word_tfidf,
    "combined_tfidf": create_combined_tfidf_embeddings(full_word_tfidf, full_tfidf),
    "bert": bert_embeddings,
    "glove": glove_embeddings,
}
results_path = RESULTS_DIR / "results.csv"
best_run, final_results = save_best_classifier_results(
    performance_comparison, classifier_results, full_features,
    embedding_metadata, results_path,
)


In [ ]:
print(f"Best classifier: {best_run['classifier']} | {EMBEDDING_LABELS[best_run['embedding']]}")
print(f"PR AUC: {best_run['pr_auc']:,.3f} | ROC AUC: {best_run['roc_auc']:,.3f}")
print(f"Saved {len(final_results):,} predictions to {results_path}")
display(final_results.head(10))
